In [ ]:
# =========================================================
# CELL 1: Import the required Python packages
# =========================================================
#
# This notebook is designed for Google Colab.
#
# The line below installs/updates gdown.
# gdown is a small Python package that allows us to download
# files or folders directly from a shared Google Drive link.
!pip -q install -U gdown

# NumPy is used for numerical calculations and arrays.
import numpy as np

# Matplotlib is used to create plots and figures.
import matplotlib.pyplot as plt

# pandas is used to read and organize tabular data.
import pandas as pd

# scipy.stats provides statistical tools such as
# Gaussian kernel density estimation.
from scipy import stats

# scipy.optimize provides numerical optimization tools.
# We will use it later to estimate the mode.
from scipy import optimize

# Path helps us work with file and folder paths.
from pathlib import Path

# shutil is used to remove an old downloaded folder
# if this cell is run more than once.
import shutil

# gdown is used to download the shared Google Drive folder.
import gdown

In [ ]:
# =========================================================
# CELL 2: Download the class data file directly from Google Drive
# =========================================================
#
# The data file is shared as:
# "Anyone with the link" -> "Viewer"
#
# We use the Google Drive FILE ID directly instead of the
# full sharing URL. This is more reliable with gdown.

# File ID taken from the shared Google Drive link:
#
# https://drive.google.com/file/d/1BnTZ1L6h63wYpyGJ8dPRmgdfL7zU_ebC/view?usp=sharing
file_id = "1BnTZ1L6h63wYpyGJ8dPRmgdfL7zU_ebC"

# This is the name and location that Colab will use
# for the downloaded data file.
DATA_FILE = "/content/fake_timestream_10min.txt"

# Download the file directly using its Google Drive file ID.
#
# id=file_id
#     tells gdown exactly which Google Drive file to download.
#
# output=DATA_FILE
#     saves the downloaded file with this name inside Colab.
#
# quiet=False
#     shows the download progress.
download_result = gdown.download(
    id=file_id,
    output=DATA_FILE,
    quiet=False
)

# Check whether gdown returned a valid download path
# and whether the file actually exists in Colab.
if download_result is None or not Path(DATA_FILE).exists():
    raise FileNotFoundError(
        "The data file could not be downloaded. "
        "Please confirm that it is shared as "
        "'Anyone with the link - Viewer'."
    )

# Print the file path so students can confirm
# that the data file is ready.
print("Data file downloaded successfully.")
print("Data file path:")
print(DATA_FILE)

In [ ]:
# =========================================================
# CELL 3: Define helper functions used later
# =========================================================

# ---------------------------------------------------------
# Function 1: Moving-average / boxcar smoothing
# ---------------------------------------------------------
#
# This function takes:
# data        -> the numerical array that we want to smooth
# window_size -> number of neighboring points included
#                in each moving average
def boxcar_smooth(data, window_size):

    # Create a window containing equal weights.
    #
    # For example, if window_size = 5,
    # this produces [0.2, 0.2, 0.2, 0.2, 0.2].
    window = np.ones(window_size) / window_size

    # np.convolve slides the averaging window across the data.
    # mode="same" keeps the output array the same length
    # as the original input array.
    smoothed_data = np.convolve(
        data,
        window,
        mode="same"
    )

    # Return the smoothed data to the user.
    return smoothed_data


# ---------------------------------------------------------
# Function 2: Gaussian probability-density function
# ---------------------------------------------------------
#
# x   -> position where the Gaussian is evaluated
# mu  -> mean of the Gaussian distribution
# sig -> standard deviation of the Gaussian distribution
def gaussian(x, mu, sig):

    # This is the mathematical Gaussian probability-density
    # function:
    #
    # p(x) = 1/(sqrt(2*pi)*sig)
    #        * exp[-(x-mu)^2/(2*sig^2)]
    return (
        1.0
        / (np.sqrt(2.0 * np.pi) * sig)
        * np.exp(
            -np.power((x - mu) / sig, 2.0) / 2.0
        )
    )


# ---------------------------------------------------------
# Function 3: Estimate the mode of a distribution
# ---------------------------------------------------------
#
# data         -> original numerical data
# distribution -> estimated probability-density function
def calc_minimize_mode(data, distribution):

    # The mode occurs where the probability density is largest.
    #
    # scipy.optimize.minimize searches for a MINIMUM,
    # so we minimize 1/pdf(x). The maximum of pdf(x)
    # corresponds to the minimum of 1/pdf(x).
    def objective(x):
        return 1 / distribution.pdf(x)[0]

    # Restrict the search to values between the minimum
    # and maximum of the observed data.
    bounds = [(min(data), max(data))]

    # Search numerically for the value that minimizes
    # the objective function.
    solution = optimize.minimize(
        objective,
        [1],
        bounds=bounds
    )

    # solution.x contains the estimated x-position of the mode.
    return solution.x[0]

In [ ]:
# =========================================================
# CELL 4: Read the data file and add simulated Gaussian noise
# =========================================================

# Fix the random seed.
#
# Random-number generators normally produce a different
# sequence every time the notebook runs.
#
# Setting a fixed seed makes the random-noise realization
# reproducible, so students should obtain the same result.
random_seed = 16
np.random.seed(seed=random_seed)

# Read the tab-separated text file using pandas.
#
# DATA_FILE is the path found in Cell 2.
#
# header=1 tells pandas to use the SECOND row of the text file
# as the column-header row.
#
# delimiter="\t" means that columns are separated by tabs.
raw_data = pd.read_csv(
    DATA_FILE,
    header=1,
    delimiter="\t"
)

# raw_data.values converts the pandas table into
# a NumPy-style numerical array.
#
# [:, 0] means:
# take all rows from column 0.
#
# The first column contains time.
time = np.array(raw_data.values[:, 0])

# [:, 31] means:
# take all rows from column 31.
#
# This is the signal column used in the original class code.
data = np.array(raw_data.values[:, 31])

# len(data) gives the total number of measurements.
npts = len(data)

# Generate artificial Gaussian random noise.
#
# loc=50   -> mean of the Gaussian distribution
# scale=10 -> standard deviation
# size=npts -> generate the same number of noise values
#              as there are data points
noise = np.random.normal(
    loc=50,
    scale=10,
    size=npts
)

# Add the simulated noise to the original signal.
#
# NumPy adds the two arrays element by element.
dpn = data + noise

# Print a few basic checks.
print("Number of data points:", npts)
print("Minimum time:", np.min(time))
print("Maximum time:", np.max(time))

In [ ]:
# =========================================================
# CELL 5: Plot the signal after adding Gaussian noise
# =========================================================

# Create a new figure.
# figsize=(8,5) controls the width and height of the figure.
fig = plt.figure(figsize=(8, 5))

# Add one plotting area to the figure.
# (1,1,1) means:
# 1 row, 1 column, use the first plotting position.
ax = fig.add_subplot(1, 1, 1)

# Plot time on the x-axis and the noisy signal on the y-axis.
ax.plot(time, dpn)

# Add axis labels.
ax.set_xlabel("Time")
ax.set_ylabel("Amplitude")

# Add a title.
ax.set_title("Simulated Time Stream")

# Display the figure.
plt.show()

In [ ]:
# =========================================================
# CELL 6: Plot the time stream and histogram side by side
# =========================================================

# Create two side-by-side plotting areas.
#
# width_ratios=[3,1] makes the time-series panel
# three times wider than the histogram panel.
fig, ax = plt.subplots(
    1,
    2,
    gridspec_kw={"width_ratios": [3, 1]},
    figsize=(10, 5)
)

# ---------------------------------------------------------
# Left panel: time-series data
# ---------------------------------------------------------

# Plot the noisy signal as a function of time.
ax[0].plot(time, dpn)

# Label the axes.
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Amplitude")


# ---------------------------------------------------------
# Right panel: histogram
# ---------------------------------------------------------

# Automatically choose reasonable histogram-bin edges
# based on the distribution of the noisy signal.
bins = np.histogram_bin_edges(
    dpn,
    bins="auto"
)

# Count how many data points fall into each histogram bin.
#
# hist_0 contains the number of points in each bin.
# The second output contains the bin edges, which we
# already stored in the variable bins.
hist_0, _ = np.histogram(
    dpn,
    bins=bins
)

# Calculate the center of every histogram bin.
#
# Example:
# if a bin runs from 10 to 12,
# its center is (10 + 12)/2 = 11.
xcenters = (bins[:-1] + bins[1:]) / 2

# Plot histogram frequency on the horizontal axis
# and amplitude-bin centers on the vertical axis.
ax[1].step(
    hist_0,
    xcenters
)

# Label the horizontal axis of the histogram.
ax[1].set_xlabel("Frequency")

# Hide duplicate y-axis tick labels on the histogram panel.
ax[1].tick_params(
    axis="y",
    labelleft=False
)

# Remove the horizontal space between the two panels.
fig.subplots_adjust(wspace=0)

# Display the figure.
plt.show()

In [ ]:
# =========================================================
# CELL 7: Plot the histogram by itself
# =========================================================

# Create a new figure.
fig = plt.figure(figsize=(8, 8))

# Add one plotting area.
ax = fig.add_subplot(1, 1, 1)

# Plot amplitude-bin centers on the x-axis
# and histogram counts on the y-axis.
ax.step(
    xcenters,
    hist_0
)

# Add axis labels.
ax.set_xlabel("Amplitude")
ax.set_ylabel("Frequency")

# Add a descriptive title.
ax.set_title("Histogram of the Simulated Time Stream")

# Display the figure.
plt.show()

In [ ]:
# =========================================================
# CELL 8: Examine the Gaussian noise by itself
# =========================================================

# Create two side-by-side panels.
fig, ax = plt.subplots(
    1,
    2,
    gridspec_kw={"width_ratios": [3, 1]},
    figsize=(10, 5)
)

# ---------------------------------------------------------
# Left panel: Gaussian noise versus time
# ---------------------------------------------------------

# Plot only the artificial Gaussian noise.
ax[0].plot(
    time,
    noise
)

# Add labels.
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Amplitude")


# ---------------------------------------------------------
# Right panel: histogram of the Gaussian noise
# ---------------------------------------------------------

# Choose histogram-bin edges automatically.
bins = np.histogram_bin_edges(
    noise,
    bins="auto"
)

# Count the number of noise values in each bin.
hist_0, _ = np.histogram(
    noise,
    bins=bins
)

# Calculate the center of each histogram bin.
xcenters = (bins[:-1] + bins[1:]) / 2

# Plot frequency horizontally and amplitude vertically.
ax[1].step(
    hist_0,
    xcenters
)

# Add x-axis label.
ax[1].set_xlabel("Frequency")

# Hide duplicate vertical-axis labels.
ax[1].tick_params(
    axis="y",
    labelleft=False
)

# Remove spacing between the two panels.
fig.subplots_adjust(wspace=0)

# Display the plot.
plt.show()

In [ ]:
# =========================================================
# CELL 9: Compare the noise histogram with a Gaussian model
# =========================================================

# Create a figure with two panels.
fig = plt.figure(figsize=(10, 5))


# ---------------------------------------------------------
# Left panel: histogram of Gaussian noise
# ---------------------------------------------------------

ax = fig.add_subplot(1, 2, 1)

# Plot histogram counts versus amplitude-bin centers.
ax.step(
    xcenters,
    hist_0
)

# Label axes.
ax.set_xlabel("Amplitude")
ax.set_ylabel("Frequency")


# ---------------------------------------------------------
# Right panel: histogram + theoretical Gaussian curve
# ---------------------------------------------------------

ax = fig.add_subplot(1, 2, 2)

# Plot the same histogram.
ax.step(
    xcenters,
    hist_0
)

# Create 100 x-values between 0 and 100.
# These points will be used to draw a smooth Gaussian curve.
xpts = np.linspace(
    0,
    100,
    100
)

# Evaluate a Gaussian with:
# mean = 50
# standard deviation = 10
#
# These are the same values that were used earlier
# when generating the simulated random noise.
ypts = gaussian(
    xpts,
    50,
    10
)

# gaussian() returns a probability density, whereas hist_0
# contains actual histogram counts.
#
# Therefore, scale the probability-density curve so that
# it can be visually compared with the histogram counts.
#
# np.sum(hist_0)      -> total number of observations
# bins[1] - bins[0]   -> histogram-bin width
ypts = (
    np.sum(hist_0)
    * ypts
    * (bins[1] - bins[0])
)

# Plot the theoretical Gaussian curve using a dashed line.
ax.plot(
    xpts,
    ypts,
    linestyle="--"
)

# Add the x-axis label.
ax.set_xlabel("Amplitude")

# Adjust spacing between panels.
fig.subplots_adjust(wspace=0.12)

# Display the figure.
plt.show()

In [ ]:
# =========================================================
# CELL 10: Estimate the mode of the Gaussian-noise data
# =========================================================

# Create a figure.
fig = plt.figure(figsize=(8, 8))

# Add one plotting area.
ax = fig.add_subplot(1, 1, 1)

# Plot the histogram.
ax.step(
    xcenters,
    hist_0
)

# Add labels.
ax.set_xlabel("Amplitude")
ax.set_ylabel("Frequency")

# Estimate a smooth probability-density function from the data.
#
# gaussian_kde means Gaussian Kernel Density Estimation.
# Instead of relying only on histogram-bin locations,
# KDE gives us a smooth estimate of the underlying distribution.
distribution = stats.gaussian_kde(noise)

# Find the x-value where the estimated probability density
# reaches its maximum. This is our estimate of the mode.
thismode = calc_minimize_mode(
    noise,
    distribution
)

# Print the calculated mode.
print(f"Mode is: {thismode}")

# Draw a vertical line at the estimated mode.
ax.axvline(thismode)

# Display the figure.
plt.show()

In [ ]:
# =========================================================
# CELL 11: Calculate the median
# =========================================================

# Create a figure.
fig = plt.figure(figsize=(8, 8))

# Add one plotting area.
ax = fig.add_subplot(1, 1, 1)

# Plot the histogram.
ax.step(
    xcenters,
    hist_0
)

# Label the axes.
ax.set_xlabel("Amplitude")
ax.set_ylabel("Frequency")

# np.median finds the middle value of the ordered data.
#
# Half of the data lie below the median
# and half lie above the median.
thismedian = np.median(noise)

# Print the median.
print(f"Median is: {thismedian}")

# Draw a vertical line at the median.
ax.axvline(thismedian)

# Display the figure.
plt.show()

In [ ]:
# =========================================================
# CELL 12: Calculate the mean
# =========================================================

# Create a figure.
fig = plt.figure(figsize=(8, 8))

# Add one plotting area.
ax = fig.add_subplot(1, 1, 1)

# Plot the histogram.
ax.step(
    xcenters,
    hist_0
)

# Add axis labels.
ax.set_xlabel("Amplitude")
ax.set_ylabel("Frequency")

# np.mean calculates the arithmetic average:
#
# mean = sum of all values / number of values
thismean = np.mean(noise)

# Print the mean.
print(f"Mean is: {thismean}")

# Draw a vertical line at the mean.
ax.axvline(thismean)

# Display the figure.
plt.show()

In [ ]:
# =========================================================
# CELL 13: Calculate the standard deviation
# =========================================================

# Create a figure.
fig = plt.figure(figsize=(8, 8))

# Add one plotting area.
ax = fig.add_subplot(1, 1, 1)

# Plot the histogram.
ax.step(
    xcenters,
    hist_0
)

# Label the axes.
ax.set_xlabel("Amplitude")
ax.set_ylabel("Frequency")

# Draw a vertical line at the mean calculated earlier.
ax.axvline(thismean)

# np.std calculates the standard deviation.
#
# Standard deviation measures the typical spread
# of the values around the mean.
thisstd = np.std(noise)

# Print the standard deviation.
print(f"STD is: {thisstd}")

# Draw vertical dotted lines at:
#
# mean + 1 standard deviation
# mean - 1 standard deviation
ax.axvline(
    thismean + thisstd,
    linestyle=":"
)

ax.axvline(
    thismean - thisstd,
    linestyle=":"
)

# Display the figure.
plt.show()

In [ ]:
# =========================================================
# CELL 14: Calculate HWHM and FWHM
# =========================================================
#
# For a Gaussian distribution:
#
# FWHM approximately equals 2.35 * sigma
#
# where sigma is the standard deviation.
#
# HWHM means Half Width at Half Maximum:
#
# HWHM = FWHM / 2

# Create a figure.
fig = plt.figure(figsize=(8, 8))

# Add one plotting area.
ax = fig.add_subplot(1, 1, 1)

# Plot the histogram.
ax.step(
    xcenters,
    hist_0
)

# Add labels.
ax.set_xlabel("Amplitude")
ax.set_ylabel("Frequency")

# Draw a vertical line at the mean.
ax.axvline(thismean)

# Calculate Half Width at Half Maximum.
thishwhm = thisstd * 2.35 / 2.0

# FWHM is twice the HWHM.
thisfwhm = 2 * thishwhm

# Print both quantities.
print(f"HWHM is: {thishwhm}")
print(f"FWHM is: {thisfwhm}")

# Mark mean + HWHM.
ax.axvline(
    thismean + thishwhm,
    linestyle=":"
)

# Mark mean - HWHM.
ax.axvline(
    thismean - thishwhm,
    linestyle=":"
)

# Display the figure.
plt.show()

In [ ]:
# =========================================================
# CELL 15: Final comparison
# =========================================================
#
# In this figure we compare:
#
# 1. the simulated Gaussian-noise time series,
# 2. the histogram of the noise,
# 3. the Gaussian model used to generate the noise,
# 4. a Gaussian model estimated from the measured
#    mean and standard deviation.

# Create two panels.
fig, ax = plt.subplots(
    1,
    2,
    gridspec_kw={"width_ratios": [2, 2]},
    figsize=(10, 5)
)

# ---------------------------------------------------------
# Left panel: Gaussian-noise time series
# ---------------------------------------------------------

# Plot noise as a function of time.
ax[0].plot(
    time,
    noise
)

# Add labels.
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Amplitude")


# ---------------------------------------------------------
# Right panel: histogram and Gaussian models
# ---------------------------------------------------------

# Recalculate histogram-bin edges.
bins = np.histogram_bin_edges(
    noise,
    bins="auto"
)

# Count values in every histogram bin.
hist_0, _ = np.histogram(
    noise,
    bins=bins
)

# Calculate the center of each bin.
xcenters = (bins[:-1] + bins[1:]) / 2

# Plot the measured data histogram.
ax[1].step(
    hist_0,
    xcenters,
    label="Data Histogram"
)

# Plot the input Gaussian model.
#
# ypts was calculated earlier using:
# mean = 50
# standard deviation = 10
ax[1].plot(
    ypts,
    xpts,
    linestyle="--",
    label="Input Model"
)

# Now calculate a Gaussian model using values
# estimated directly from the simulated noise data.
#
# np.mean(noise) gives the measured mean.
# np.std(noise) gives the measured standard deviation.
ypts2 = gaussian(
    xpts,
    np.mean(noise),
    np.std(noise)
)

# Scale this probability-density curve to histogram counts.
ypts2 = (
    np.sum(hist_0)
    * ypts2
    * (bins[1] - bins[0])
)

# Plot the Gaussian model estimated from the data.
ax[1].plot(
    ypts2,
    xpts,
    linestyle="--",
    label="Estimated Model"
)

# Label the horizontal axis.
ax[1].set_xlabel("Frequency")

# Hide duplicate y-axis labels.
ax[1].tick_params(
    axis="y",
    labelleft=False
)

# Show the labels identifying the three curves.
ax[1].legend()

# Remove space between the two panels.
fig.subplots_adjust(wspace=0)

# Display the final figure.
plt.show()